# Learned unicycle dynamics

Same Robotarium robot 11 split as the rest of this folder. The map is the true plant: commanded body velocity to what was actualized, then SE(2).

State is $g=(x,y,\theta)$ and $\nu=(v,\omega)$. Input is the logged command $u=(v_c,\omega_c)$ after saturation and barrier filtering.

$$\nu_{k+1} = u_k + f_\theta(\cdot)$$
$$g_{k+1} = g_k \circ \operatorname{Exp}(\Delta t\,\nu_{k+1})$$

$f_\theta$ is zero at initialization, so epoch 0 is $\nu_{k+1}=u_k$. Three models, same trunk, different what $f_\theta$ is allowed to see:

1. **Full state.** $f_\theta(x,y,\theta,v,\omega,v_c,\omega_c)$. Not SE(2) invariant.
2. **Quotient.** $f_\theta(v,\omega,v_c,\omega_c)$. Body-frame only.
3. **Quotient + symmetry.** Same inputs, left–right (longitudinal-axis) parity. $v$ even, $\omega$ odd. No front–back constraint.

Loss is a 5 s open-loop Vicon $xy$ error plus encoder $(v,\omega)$. The checkpoint is the lowest full validation position RMSE.

First comparison is full state vs quotient. Next is quotient vs quotient + left–right (longitudinal-axis) symmetry. Each pair: training curves, then two held-out test overheads (Vicon, commanded unicycle, learned path), then RMSE.


In [1]:
import sys
from pathlib import Path

import torch

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "src" / "learned_dynamics").exists() else HERE.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from unicycle.data import load_split
from unicycle.figures import COLORS, plot_dynamics_rollouts, plot_loss
from unicycle.models import DynamicsFullState, DynamicsQuotient, DynamicsQuotientSymmetry
from unicycle.rollout import held_out_rmse, pooled_rollout_position_rmse
from unicycle.train import CHECKPOINT_DIR, load_checkpoint, save_checkpoint, train_open_loop

SEED = 0
EPOCHS = 100
LR = 1e-4
PATIENCE = 10
DYNAMICS_DIR = CHECKPOINT_DIR / "dynamics"

MODELS = (
    ("dynamics_fullstate", DynamicsFullState, "full state", COLORS["fullstate"]),
    ("dynamics_quotient", DynamicsQuotient, "quotient", COLORS["quotient"]),
    ("dynamics_quotient_symmetry", DynamicsQuotientSymmetry, "quotient + symmetry", COLORS["quotient_symmetry"]),
)

train_logs, val_logs, test_logs = load_split()
print(f"train {sum(log.n_poses for log in train_logs)}  val {sum(log.n_poses for log in val_logs)}  test {sum(log.n_poses for log in test_logs)}")
print("test", ", ".join(log.name for log in test_logs))

train 11645  val 3887  test 5992
test figure8, lawnmower_diagonal_reverse, random_trajectory


Train or load. Checkpoints go under `runs/unicycle/dynamics/`.

In [2]:
def load_or_train(name, cls):
    path = DYNAMICS_DIR / f"{name}.pt"
    if path.is_file():
        model, result, _residual = load_checkpoint(name, folder=DYNAMICS_DIR)
        print(f"loaded {name} from {path}  best epoch {result.best_epoch}  val {result.best_error:.4f} m")
        return model, result
    torch.manual_seed(SEED)
    model = cls()
    result = train_open_loop(
        model,
        train_logs,
        val_logs,
        epochs=EPOCHS,
        lr=LR,
        rollout_every=5,
        patience=PATIENCE,
        name=name,
    )
    save_checkpoint(name, model, result, residual=False, folder=DYNAMICS_DIR)
    return model, result


fitted = {}
for name, cls, _label, _color in MODELS:
    fitted[name] = load_or_train(name, cls)

dynamics_fullstate  5 s windows  51 steps  train 11237  validation 3479  batch 64
dynamics_fullstate  epoch 00  5s mse 1.482161e-03  val 1.201886e-03  validation position RMSE 0.3644 m  saved
dynamics_fullstate  epoch 01  5s mse 1.413718e-03  val 1.178240e-03
dynamics_fullstate  epoch 02  5s mse 1.351980e-03  val 1.157274e-03
dynamics_fullstate  epoch 03  5s mse 1.319274e-03  val 1.140600e-03
dynamics_fullstate  epoch 04  5s mse 1.304940e-03  val 1.122595e-03
dynamics_fullstate  epoch 05  5s mse 1.293991e-03  val 1.119123e-03  validation position RMSE 0.1659 m  saved
dynamics_fullstate  epoch 06  5s mse 1.283854e-03  val 1.115129e-03
dynamics_fullstate  epoch 07  5s mse 1.275917e-03  val 1.109882e-03
dynamics_fullstate  epoch 08  5s mse 1.268451e-03  val 1.112158e-03
dynamics_fullstate  epoch 09  5s mse 1.262942e-03  val 1.109503e-03
dynamics_fullstate  epoch 10  5s mse 1.257033e-03  val 1.108093e-03  validation position RMSE 0.1937 m
dynamics_fullstate  epoch 11  5s mse 1.253673e-03  

Training curves, full state vs quotient. Solid is train 5 s MSE, dashed is validation. The marker is the saved full-val checkpoint.

In [ ]:
plot_loss(
    fitted["dynamics_quotient"][1],
    color=COLORS["quotient"],
    label="quotient",
    others=[(fitted["dynamics_fullstate"][1], COLORS["fullstate"], "full state")],
)

Held-out test overhead, full state. Vicon is ground truth. Dashed unicycle is the command executed exactly. Solid is the learned leftover on $(x,y,\theta,\nu,u)$.

In [ ]:
plot_dynamics_rollouts(
    test_logs,
    [("full state", fitted["dynamics_fullstate"][0], COLORS["fullstate"])],
    set_label="Test · full state",
)

Held-out test overhead, quotient. Same three traces: Vicon, commanded unicycle, leftover on $(\nu,u)$ only.

In [ ]:
plot_dynamics_rollouts(
    test_logs,
    [("quotient", fitted["dynamics_quotient"][0], COLORS["quotient"])],
    set_label="Test · quotient",
)

Held-out numbers. Full-trajectory RMSE and 5 s open-loop RMSE.

In [ ]:
print(f"{'model':24s}  {'split':>5s}  {'full':>8s}  {'5 s':>8s}  epoch")
for name, _cls, label, _color in MODELS[:2]:
    model, result = fitted[name]
    for split_name, logs in (("val", val_logs), ("test", test_logs)):
        full = held_out_rmse(model, logs)
        five = pooled_rollout_position_rmse(model, logs, horizon_s=5.0)
        print(f"{label:24s}  {split_name:>5s}  {full:8.4f}  {five:8.4f}  {result.best_epoch:5d}")
        if split_name == "test":
            for log in logs:
                one_full = held_out_rmse(model, [log])
                one_five = pooled_rollout_position_rmse(model, [log], horizon_s=5.0)
                print(f"  {log.name:40s}  full {one_full:.4f}  5s {one_five:.4f}")

Training curves, quotient vs quotient + left–right symmetry. Same leftover on $(\nu,u)$; the symmetric head forces $v$ even and $\omega$ odd across the robot's longitudinal (FLU $x$) axis.

In [ ]:
plot_loss(
    fitted["dynamics_quotient_symmetry"][1],
    color=COLORS["quotient_symmetry"],
    label="quotient + symmetry",
    others=[(fitted["dynamics_quotient"][1], COLORS["quotient"], "quotient")],
)

Held-out test overhead, quotient. Vicon, commanded unicycle, leftover on $(\nu,u)$.

In [ ]:
plot_dynamics_rollouts(
    test_logs,
    [("quotient", fitted["dynamics_quotient"][0], COLORS["quotient"])],
    set_label="Test · quotient",
)

Held-out test overhead, quotient + symmetry. Same three traces; $f_\theta$ is left–right even/odd on that one axis.

In [ ]:
plot_dynamics_rollouts(
    test_logs,
    [("quotient + symmetry", fitted["dynamics_quotient_symmetry"][0], COLORS["quotient_symmetry"])],
    set_label="Test · quotient + symmetry",
)

Held-out numbers, quotient vs quotient + symmetry.

In [ ]:
print(f"{'model':24s}  {'split':>5s}  {'full':>8s}  {'5 s':>8s}  epoch")
for name, _cls, label, _color in MODELS[1:]:
    model, result = fitted[name]
    for split_name, logs in (("val", val_logs), ("test", test_logs)):
        full = held_out_rmse(model, logs)
        five = pooled_rollout_position_rmse(model, logs, horizon_s=5.0)
        print(f"{label:24s}  {split_name:>5s}  {full:8.4f}  {five:8.4f}  {result.best_epoch:5d}")
        if split_name == "test":
            for log in logs:
                one_full = held_out_rmse(model, [log])
                one_five = pooled_rollout_position_rmse(model, [log], horizon_s=5.0)
                print(f"  {log.name:40s}  full {one_full:.4f}  5s {one_five:.4f}")